# Crop Permutation Significance

Ce notebook reprend le script `crop_permutation_significance.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste la significativite des scores crop avant usage dans une politique live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Permutation significance for crop attention/PPE predictions.
- Artefacts controles : Crop permutation significance audit exists. (`runs/exp_027_crop_permutation_significance/metrics/crop_permutation_significance.csv`).
- Dossier de sortie par defaut : `runs/exp_027_crop_permutation_significance`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_permutation_significance.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from ml_pipeline import ROOT, safe_auc, write_json


## Fonction `aggregate_video`

Cette cellule definit `aggregate_video`. Elle prepare une partie du script.

In [ ]:
def aggregate_video(df, target):
    return df.groupby("video_id", as_index=False).agg(label=(f"{target}_label", "max"), risk=("risk", "mean"))


## Fonction `parent_permutation_null`

Cette cellule definit `parent_permutation_null`. Elle prepare une partie du script.

In [ ]:
def parent_permutation_null(df, target, level, n_permutations, seed):
    rng = np.random.default_rng(seed)
    video = aggregate_video(df, target)
    video_labels = video["label"].astype(int).to_numpy()
    video_risk = video["risk"].to_numpy()
    if level == "video":
        real_y = video_labels
        real_p = video_risk
    else:
        label_by_video = dict(zip(video["video_id"], video_labels))
        real_y = df["video_id"].map(label_by_video).astype(int).to_numpy()
        real_p = df["risk"].to_numpy()
    real_ap = safe_auc(average_precision_score, real_y, real_p)
    real_auc = safe_auc(roc_auc_score, real_y, real_p)
    null_ap = []
    null_auc = []
    for _ in range(n_permutations):
        shuffled = video_labels.copy()
        rng.shuffle(shuffled)
        if level == "video":
            y = shuffled
            p = video_risk
        else:
            label_by_video = dict(zip(video["video_id"], shuffled))
            y = df["video_id"].map(label_by_video).astype(int).to_numpy()
            p = df["risk"].to_numpy()
        ap = safe_auc(average_precision_score, y, p)
        auc = safe_auc(roc_auc_score, y, p)
        if ap is not None:
            null_ap.append(ap)
        if auc is not None:
            null_auc.append(auc)
    null_ap = np.asarray(null_ap, dtype=float)
    null_auc = np.asarray(null_auc, dtype=float)
    return {
        "level": level,
        "n": int(len(real_y)),
        "positive": int(real_y.sum()),
        "prevalence": float(real_y.mean()) if len(real_y) else 0.0,
        "real_ap": real_ap,
        "real_roc_auc": real_auc,
        "null_ap_mean": float(np.mean(null_ap)) if len(null_ap) else None,
        "null_ap_std": float(np.std(null_ap)) if len(null_ap) else None,
        "null_ap_p95": float(np.quantile(null_ap, 0.95)) if len(null_ap) else None,
        "null_auc_mean": float(np.mean(null_auc)) if len(null_auc) else None,
        "null_auc_p95": float(np.quantile(null_auc, 0.95)) if len(null_auc) else None,
        "p_value_ap_ge_real": float((np.sum(null_ap >= real_ap) + 1) / (len(null_ap) + 1)) if len(null_ap) and real_ap is not None else None,
    }


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    crop_run = Path(args.crop_run)
    if not crop_run.is_absolute():
        crop_run = ROOT / crop_run
    out_dir = Path(args.out_dir)
    if not out_dir.is_absolute():
        out_dir = ROOT / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "metrics").mkdir(parents=True, exist_ok=True)
    pred = pd.read_csv(crop_run / "features" / "crop_cnn_all_predictions.csv")
    rows = []
    for (target, architecture, split), group in pred.groupby(["target", "architecture", "split"]):
        for level in ["crop", "video"]:
            row = parent_permutation_null(group, target, level, args.permutations, args.seed)
            row.update(
                {
                    "target": target,
                    "architecture": architecture,
                    "split": split,
                    "permutations": int(args.permutations),
                    "null_policy": "parent-video labels permuted within target/architecture/split",
                }
            )
            rows.append(row)
    metrics = pd.DataFrame(rows)
    metrics.to_csv(out_dir / "metrics" / "crop_permutation_significance.csv", index=False)
    write_json(
        out_dir / "config.json",
        {
            "crop_run": str(crop_run),
            "permutations": args.permutations,
            "seed": args.seed,
            "note": "Tests whether real crop predictions beat parent-label permutation null distributions.",
        },
    )

    test = metrics[metrics["split"] == "test"].sort_values(["target", "level", "real_ap"], ascending=[True, True, False])
    lines = ["# Crop Permutation Significance", ""]
    lines.append("Real crop predictions are compared against a null distribution produced by permuting parent-video labels inside each target/architecture/split. This keeps the prediction scores fixed and tests whether the reported AP is meaningfully above random parent-label assignment.")
    lines.append("")
    lines.append("## Test Split")
    lines.append("")
    lines.append("| target | level | architecture | n | pos | real AP | null AP mean | null AP p95 | p(AP_null >= real) | real AUC |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|---:|---:|")
    for _, row in test.iterrows():
        lines.append(
            f"| {row['target']} | {row['level']} | {row['architecture']} | {int(row['n'])} | {int(row['positive'])} | {row['real_ap']:.3f} | {row['null_ap_mean']:.3f} | {row['null_ap_p95']:.3f} | {row['p_value_ap_ge_real']:.3f} | {row['real_roc_auc'] if pd.notna(row['real_roc_auc']) else 'NA'} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- Low p-values mean the model's AP is unlikely under random parent-label assignment.")
    lines.append("- Attention test evidence is fragile because there is only one positive test parent in the fixed split.")
    lines.append("- This is a significance sanity check, not proof of actor/background robustness.")
    (out_dir / "crop_permutation_significance.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Permutation significance for crop attention/PPE predictions.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--out-dir", default="runs/exp_027_crop_permutation_significance")
    parser.add_argument("--permutations", type=int, default=2000)
    parser.add_argument("--seed", type=int, default=1234)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Le dossier de sortie est rendu unique pour ne pas ecraser le run existant.
from datetime import datetime
import sys

OUT_DIR_BASE = "runs/exp_027_crop_permutation_significance"
OUT_DIR = f"{OUT_DIR_BASE}_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--out-dir", OUT_DIR]

ancien_argv = sys.argv[:]
sys.argv = ["crop_permutation_significance.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
